# AutoML Phát hiện Gian lận Bảo hiểm Ô tô (v2 - đã sửa lỗi Python 3.12)

**Vấn đề đã sửa:** Google Colab hiện dùng Python 3.12, nhưng PyCaret chỉ hỗ trợ Python 3.9-3.11. Notebook này tự động tạo một môi trường Python 3.11 riêng bên trong Colab để chạy PyCaret, không cần thao tác thủ công gì thêm.

**Cách dùng:** Chạy lần lượt từng ô từ trên xuống (Shift+Enter). Khi được hỏi, upload đúng file CSV được yêu cầu.

## Bước 1: Cài đặt Python 3.11 và tạo môi trường riêng

In [ ]:
# Cài Python 3.11 (mất khoảng 1 phút)
!sudo apt-get update -qq
!sudo apt-get install -y python3.11 python3.11-venv python3.11-dev -qq

# Tạo môi trường ảo riêng dùng Python 3.11
!python3.11 -m venv /content/pycaret_env

# Cài PyCaret vào môi trường riêng này (mất khoảng 2-3 phút)
!/content/pycaret_env/bin/pip install --upgrade pip -q
!/content/pycaret_env/bin/pip install pycaret pandas -q

print("\n✅ Đã cài đặt xong môi trường Python 3.11 với PyCaret.")

## Bước 2: Upload dữ liệu huấn luyện

In [ ]:
from google.colab import files

print("Vui lòng chọn file DR_Demo_Car_Insurance_Fraud_train.csv:")
uploaded = files.upload()
train_filename = list(uploaded.keys())[0]
print(f"\n✅ Đã upload: {train_filename}")

## Bước 3: Viết script huấn luyện và chạy bằng Python 3.11

In [ ]:
train_script = f'''
import pandas as pd
from pycaret.classification import *

df = pd.read_csv("{train_filename}")

# Loai bo cac cot khong can thiet (ID va mo ta van ban gia lap)
df = df.drop(columns=["ID", "CLAIM_DESCRIPTION"], errors="ignore")

# Khoi tao PyCaret voi FRAUD la cot muc tieu
clf = setup(data=df, target="FRAUD", session_id=42)

# Chay AutoML - tu dong thu va so sanh nhieu thuat toan
best_model = compare_models()

# Hoan thien model tot nhat va luu ra file .pkl
final_model = finalize_model(best_model)
save_model(final_model, "fraud_model")

print("\\n=== HOAN TAT HUAN LUYEN ===")
'''

with open("train_model.py", "w") as f:
    f.write(train_script)

print("Đã tạo file train_model.py, bắt đầu chạy AutoML...\n")
!/content/pycaret_env/bin/python train_model.py

## Bước 4: Tải model đã huấn luyện về máy

In [ ]:
files.download('fraud_model.pkl')

## Bước 5: Upload dữ liệu mới cần dự đoán (test thử model)

In [ ]:
print("Vui lòng chọn file dữ liệu mới cần dự đoán (vd: du_lieu_gia_lap_test_model_FRAUD.csv):")
uploaded_new = files.upload()
new_filename = list(uploaded_new.keys())[0]
print(f"\n✅ Đã upload: {new_filename}")

## Bước 6: Viết script dự đoán và chạy bằng Python 3.11

In [ ]:
predict_script = f'''
import pandas as pd
from pycaret.classification import load_model, predict_model

df_new = pd.read_csv("{new_filename}")
df_predict = df_new.drop(columns=["CLAIM_DESCRIPTION"], errors="ignore")

final_model = load_model("fraud_model")
predictions = predict_model(final_model, data=df_predict)

result_view = predictions[["ID", "prediction_label", "prediction_score"]]
result_view.to_csv("ket_qua_du_doan.csv", index=False)

print(result_view.to_string(index=False))
print("\\n=== HOAN TAT DU DOAN ===")
'''

with open("predict_model.py", "w") as f:
    f.write(predict_script)

print("Đã tạo file predict_model.py, bắt đầu dự đoán...\n")
!/content/pycaret_env/bin/python predict_model.py

## Bước 7: Tải kết quả dự đoán về máy

In [ ]:
files.download('ket_qua_du_doan.csv')